## Thread vs Process

### Process :
    A process is an independent instance of a program being executed. When you open your web browser, that is a process. When you open a video player, that is a different process.
### Thread: 
    A thread is a "lightweight" unit of execution that lives inside a process. A single process can have many threads.

### GIL (GLOBAL INTERPRETER LOCK)
In most languages (like C++ or Java), threads can run on different CPU cores simultaneously to do heavy math. However, standard Python (CPython) has a Global Interpreter Lock (GIL).The GIL is like a single microphone in the "house." Even if you have five threads (people), only the person holding the microphone is allowed to execute Python code at that moment.Use Threads for I/O-bound tasks (waiting for a website to respond or a file to save). While one thread waits, it drops the "microphone" so another thread can work.Use Processes for CPU-bound tasks (calculating Pi to a billion digits). Each process gets its own Python interpreter and its own "microphone," allowing them to run on separate CPU cores.

In [4]:
# program to spawn 3 threads printing number from 1 to 10
import threading
import time
def print_numbers(thread_name):
    for i in range(10):
        time.sleep(0.01)
        print(f"{thread_name}: {i}")

t1 = threading.Thread(target = print_numbers, args=("Thread-1",))
t2 = threading.Thread(target = print_numbers, args=("Thread-2",))
t3 = threading.Thread(target = print_numbers, args=("Thread-3",))

t1.start()
t2.start()
t3.start()

t1.join()
t2.join()
t3.join()


Thread-1: 0Thread-2: 0
Thread-3: 0

Thread-2: 1
Thread-3: 1
Thread-1: 1
Thread-3: 2
Thread-1: 2
Thread-2: 2
Thread-1: 3
Thread-3: 3
Thread-2: 3
Thread-3: 4
Thread-2: 4
Thread-1: 4
Thread-3: 5
Thread-1: 5
Thread-2: 5
Thread-3: 6
Thread-1: 6
Thread-2: 6
Thread-3: 7
Thread-1: 7
Thread-2: 7
Thread-3: 8
Thread-2: 8
Thread-1: 8
Thread-2: 9
Thread-3: 9
Thread-1: 9


## Race Condition
A race condition occurs when the timing or order of events affects the correctness of a program. In multi-threaded programming, this usually happens when two or more threads access shared data and try to change it at the same time.

Because the threads are "racing" to finish their task, the final state of the data depends on which thread wins the race, often leading to unpredictable bugs.

## Threading Lock
A Lock (or Mutex) acts as a "talking stick." Only the thread holding the lock is allowed to modify the protected data. Others must wait in line.

In [10]:
import threading
database = 0
def increase():
    global database 
    for _ in range(100000):
        temp = database
        temp+=1
        time.sleep(0)
        database = temp
    
print(f"Starting Value: {database}")
thread1 = threading.Thread(target = increase)
thread2 = threading.Thread(target = increase)

thread1.start()
thread2.start()

thread1.join()
thread2.join()

print(f"End Value: {database}")

Starting Value: 0
End Value: 100000


In [11]:
import threading
database = 0
thread_lock = threading.Lock()
def increase():
    global database 
    for _ in range(100000):
        with thread_lock:
            temp = database
            temp+=1
            time.sleep(0)
            database = temp
    
print(f"Starting Value: {database}")
thread1 = threading.Thread(target = increase)
thread2 = threading.Thread(target = increase)

thread1.start()
thread2.start()

thread1.join()
thread2.join()

print(f"End Value: {database}")

Starting Value: 0
End Value: 200000


## What is a Deadlock?
A deadlock happens when two or more threads are permanently stuck waiting for each other to release resources.

Analogy: Person A has the paper but needs the pen. Person B has the pen but needs the paper. Neither will yield, so nothing gets signed.

### How it Happens (Circular Wait)
Thread A acquires Lock 1, then waits for Lock 2.

Thread B acquires Lock 2, then waits for Lock 1.

Both threads are now blocked infinitely.

### The Fix: Lock Ordering
To prevent this, enforce a global rule: All threads must acquire locks in the exact same order.

The New Flow: You mandate that every thread must ask for Lock 1 before Lock 2.

The Result: Thread A grabs Lock 1. Thread B tries to grab Lock 1 but is forced to wait. Thread A can freely grab Lock 2, finish its work, and release both locks. Thread B wakes up and safely proceeds. No collisions!

### Q- Implement a thread-safe stack using `threading.Lock()` and Test it with 4 threads doing push/pop concurrently

In [11]:
import threading
import time
import random
class ThreadSafeStack:
    def __init__(self):
        self.stack = []
        self.lock = threading.Lock()
    
    def push(self, val):
        with self.lock:
            self.stack.append(val)
    
    def pop(self):
        with self.lock:
            if not self.stack:
                return None
            return self.stack.pop
    
    def size(self):
        with self.lock:
            return len(self.stack)

In [12]:
def worker(stack, thread_id):
    for i in range(50):
        val = f"T{thread_id}-{i}"
        stack.push(val)
        # Slight sleep to encourage thread switching/contention
        time.sleep(random.uniform(0.001, 0.01))
    
    for _ in range(50):
        stack.pop()
        time.sleep(random.uniform(0.001, 0.01))

def run_test():
    my_stack = ThreadSafeStack()
    threads = []
    
    print("Starting 4 threads...")
    for i in range(4):
        t = threading.Thread(target=worker, args=(my_stack, i))
        threads.append(t)
        t.start()

    # Wait for all threads to finish
    for t in threads:
        t.join()

    print(f"Final Stack Size: {my_stack.size()}")
    if my_stack.size() == 0:
        print("Success: Stack is empty after equal pushes and pops!")
    else:
        print("Failure: Stack size is inconsistent!")

if __name__ == "__main__":
    run_test()

Starting 4 threads...
Final Stack Size: 200
Failure: Stack size is inconsistent!


### IMplementing thread-safe LRU Cache

In [14]:
import threading
class Node:
    def __init__(self, key, val):
        self.key = key
        self.val = val
        self.prev = None
        self.next = None

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.cache = {}
        self.lock = threading.RLock()

        self.head = Node(0,0)
        self.tail = Node(0,0)
        self.head.next = self.tail
        self.tail.prev = self.head

    def remove(self, node):
        prv, nxt = node.prev, node.next
        prv.next = nxt
        next.prev = prv
    
    def add_to_front(self, node):
        node.prev = self.head
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node

    def get(self, key: int) -> int:
        with self.lock:
            if key not in self.cache:
                return -1
            node = self.cache[key]
            self.remove(node)
            self.add_to_front(node)
            return node.val
        
    def put(self, key, value):
        with self.lock:
            if key in self.cache:
                self.remove(self.cache[key])
            node = Node(key, value)
            self.cache[key] = node
            self.add_to_front(node)
            if len(self.cache) > self.capacity:
                lru = self.tail.prev
                self.remove(lru)
                del self.cache[lru.key]


### Concurrency: threading.Condition() — wait/notify
When to use it?
Use Condition when threads must wait for a state change (not just mutual exclusion).
Example: consumers wait until buffer has items; producers wait until buffer has space.
Core contract - 
- Condition wraps a lock.
- You must hold that lock when calling wait(), notify(), notify_all().
- wait() atomically:
1) releases the lock,
2) sleeps,
3) re-acquires lock before returning.
Always wait in a while loop (while not predicate: cond.wait()) to handle spurious wakeups / races.

In [15]:
import threading
import time
import random
from collections import deque

In [22]:
class BoundedBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.q = deque()
        self.cond = threading.Condition()
    
    def put(self, item):
        with self.cond:
            while len(self.q)>=self.capacity:
                self.cond.wait() # wait until consumer remvoes something
            self.q.append(item)
            print(f"[PUT ] {item}, size={len(self.q)}")
            self.cond.notify()  # wake one waiter (likely a consumer)

    def get(self):
        with self.cond:
          while not self.q:
              self.cond.wait()  # wait until producer adds something
          item = self.q.popleft()
          print(f"[GET ] {item}, size={len(self.q)}")
          self.cond.notify()  # wake one waiter (likely a producer)
          return item
    
def producer(buf: BoundedBuffer, n=10):
    for i in range(n):
        time.sleep(random.uniform(0.05, 0.2))
        buf.put(f"item-{i}")

def consumer(buf: BoundedBuffer, n=10):
    for _ in range(10):
        time.sleep(random.uniform(0.1, 0.25))
        _ = buf.get()

buf = BoundedBuffer(capacity = 10)
t1 = threading.Thread(target=producer, args=(buf, 12))
t3 = threading.Thread(target=producer, args=(buf, 12))
t5 = threading.Thread(target=producer, args=(buf, 12))
t2 = threading.Thread(target=consumer, args=(buf, 12))
t4 = threading.Thread(target=consumer, args=(buf, 12))

t1.start()
t2.start()
t3.start()
t4.start()
t5.start()
t1.join()
t2.join()
t3.join()
t4.join()
t5.join()
print("Done")


[PUT ] item-0, size=1
[PUT ] item-0, size=2
[GET ] item-0, size=1
[PUT ] item-0, size=2
[GET ] item-0, size=1
[PUT ] item-1, size=2
[PUT ] item-1, size=3
[PUT ] item-1, size=4
[PUT ] item-2, size=5
[GET ] item-0, size=4
[PUT ] item-2, size=5
[PUT ] item-2, size=6
[PUT ] item-3, size=7
[PUT ] item-4, size=8
[PUT ] item-3, size=9
[GET ] item-1, size=8
[PUT ] item-3, size=9
[GET ] item-1, size=8
[GET ] item-1, size=7
[PUT ] item-4, size=8
[PUT ] item-4, size=9
[PUT ] item-5, size=10
[GET ] item-2, size=9
[PUT ] item-5, size=10
[GET ] item-2, size=9
[PUT ] item-5, size=10
[GET ] item-2, size=9
[PUT ] item-6, size=10
[GET ] item-3, size=9
[PUT ] item-6, size=10
[GET ] item-4, size=9
[PUT ] item-6, size=10
[GET ] item-3, size=9
[PUT ] item-7, size=10
[GET ] item-3, size=9
[PUT ] item-7, size=10
[GET ] item-4, size=9
[PUT ] item-8, size=10
[GET ] item-4, size=9
[PUT ] item-7, size=10
[GET ] item-5, size=9
[PUT ] item-8, size=10
[GET ] item-5, size=9
[PUT ] item-8, size=10
[GET ] item-5, size=

KeyboardInterrupt: 

## Reader Writer Problem

### Here are the strict rules we must enforce:

Multiple readers can read from the shared resource simultaneously (reading doesn't change the data, so it's safe).

Only one writer can write to the shared resource at a time.

No readers can read while a writer is writing (to prevent reading half-written or corrupt data).
To solve this in Python, we need to carefully manage access using synchronization primitives like threading.Lock, threading.RLock, or threading.Semaphore.

The Strategy
To allow multiple concurrent readers but exclusive writers, we need three things:

readers_count: An integer to keep track of how many readers are currently reading.

read_lock (Mutex): A lock to protect the readers_count variable so multiple threads don't scramble the count.

write_lock (Semaphore or Lock): A lock to protect the actual shared resource.

The Magic Trick: * The first reader to arrive acquires the write_lock, effectively locking out all writers.

Subsequent readers can just join in without checking the write_lock.

The last reader to leave releases the write_lock, signaling that writers can now safely access the resource.

In [28]:
import threading
import time
import random

class ReaderWriter:
    def __init__(self):
        self.shared_data =0

        self.reader_count = 0 
        self.read_lock = threading.Lock()
        self.write_lock = threading.Lock()

    def read(self, reader_id):
        with self.read_lock:
            self.reader_count += 1
            if self.reader_count == 1:
                self.write_lock.acquire()
        
        print(f"[Reader {reader_id}] Started reading. Data = {self.shared_data}. (Total readers: {self.reader_count})")
        time.sleep(random.uniform(0.1, 0.5)) # Simulate reading time
        print(f"[Reader {reader_id}] Finished reading.")

        with self.read_lock:
            self.reader_count -= 1
            if self.reader_count == 0:
                self.write_lock.release()
            
    def write(self, writer_id):
        self.write_lock.acquire()
        try:
            print(f"  -> [Writer {writer_id}] Started writing. Locking everyone out.")
            time.sleep(random.uniform(0.5, 1.0)) # Simulate writing time
            self.shared_data += 1
            print(f"  -> [Writer {writer_id}] Finished writing. New Data = {self.shared_data}.")
        finally:
            # Release the write lock
            self.write_lock.release()



In [29]:
rw = ReaderWriter()
threads = [] 
for i in range(5):
    t_read = threading.Thread(target = rw.read, args= (i+1,))
    threads.append(t_read)

    if i%2 == 0:
        t_write = threading.Thread(target = rw.write, args=(i+1,))
        threads.append(t_write)

for t in threads:
    t.start()
    time.sleep(0.05)

for t in threads:
    t.join()

print("Simulation complete")

[Reader 1] Started reading. Data = 0. (Total readers: 1)
[Reader 2] Started reading. Data = 0. (Total readers: 2)
[Reader 3] Started reading. Data = 0. (Total readers: 3)
[Reader 1] Finished reading.
[Reader 4] Started reading. Data = 0. (Total readers: 3)
[Reader 5] Started reading. Data = 0. (Total readers: 4)
[Reader 2] Finished reading.
[Reader 3] Finished reading.
[Reader 4] Finished reading.
[Reader 5] Finished reading.
  -> [Writer 1] Started writing. Locking everyone out.
  -> [Writer 1] Finished writing. New Data = 1.
  -> [Writer 3] Started writing. Locking everyone out.
  -> [Writer 3] Finished writing. New Data = 2.
  -> [Writer 5] Started writing. Locking everyone out.
  -> [Writer 5] Finished writing. New Data = 3.
Simulation complete


## IMplementing Parallel BFS

In [48]:
import concurrent.futures
import threading
import time

def parallel_bfs(graph, start_node):
    # Track the shortest distance to each node and protect it with a lock
    distances = {start_node: 0}
    distances_lock = threading.Lock()
    
    # The current level of nodes we are exploring
    frontier = [start_node]
    current_level = 0
    
    # Initialize the thread pool
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        
        while frontier:
            print(f"Level {current_level} (Nodes: {len(frontier)}): {frontier}")
            
            next_frontier = []
            next_frontier_lock = threading.Lock()

            # The worker function executed by the threads
            def explore_neighbors(node):
                local_next_frontier = []
                
                # Look at all neighbors of the assigned node
                for neighbor in graph.get(node, []):
                    # Safely check and update the global visited/distances state
                    with distances_lock:
                        if neighbor not in distances:
                            distances[neighbor] = current_level + 1
                            local_next_frontier.append(neighbor)
                
                # Safely merge the locally discovered nodes into the global next frontier
                if local_next_frontier:
                    with next_frontier_lock:
                        next_frontier.extend(local_next_frontier)

            # executor.map assigns the explore_neighbors function to every node in the frontier.
            # Wrapping it in list() blocks execution until ALL threads finish the current level.
            # This is our crucial Level-Synchronization barrier!
            list(executor.map(explore_neighbors, frontier))
            
            # Move to the next level
            frontier = next_frontier
            current_level += 1

    return distances

In [49]:
sample_graph = {
        'A': ['B', 'C', 'D'],
        'B': ['A', 'E', 'F'],
        'C': ['A', 'G'],
        'D': ['A', 'H'],
        'E': ['B', 'I'],
        'F': ['B'],
        'G': ['C', 'J', 'K'],
        'H': ['D'],
        'I': ['E'],
        'J': ['G'],
        'K': ['G']
    }

print("Starting Parallel BFS...")
start_time = time.time()

result_distances = parallel_bfs(sample_graph, 'A')

print("\nTraversal Complete!")
print(f"Distances from 'A': {result_distances}")

Starting Parallel BFS...
Level 0 (Nodes: 1): ['A']
Level 1 (Nodes: 3): ['B', 'C', 'D']
Level 2 (Nodes: 4): ['E', 'F', 'G', 'H']
Level 3 (Nodes: 3): ['I', 'J', 'K']

Traversal Complete!
Distances from 'A': {'A': 0, 'B': 1, 'C': 1, 'D': 1, 'E': 2, 'F': 2, 'G': 2, 'H': 2, 'I': 3, 'J': 3, 'K': 3}


## Dining Philosopher Problem

The Dining Philosophers problem is a classic computer science thought experiment that perfectly illustrates the dangers of poorly managed concurrency. It visualizes a scenario where five philosophers sit around a table with only five forks between them, yet each requires two forks to eat. This setup mimics software systems where multiple distinct processes (the philosophers) must share limited, mutually exclusive resources (the forks) to complete their tasks. If the system lacks strict rules for how these resources are acquired, it inevitably faces deadlock—where every process grabs one resource, waits forever for a second one, and the whole system freezes—or starvation, where a process is perpetually denied the resources it needs to ever execute

In [58]:
import threading
import time
import random

class Philosopher(threading.Thread):
    def __init__(self, index, forks):
        super().__init__(name=f"Philosopher-{index}")
        self.index = index
        
        # Determine the indices of the required forks
        left_fork_index = index
        right_fork_index = (index + 1) % len(forks)
        
        # THE FIX: Resource Hierarchy
        # Break the symmetry by forcing everyone to grab the lower-indexed lock first.
        # For Philosopher 4, left is 4, right is 0. They will grab 0 first, then 4.
        self.first_fork = forks[min(left_fork_index, right_fork_index)]
        self.second_fork = forks[max(left_fork_index, right_fork_index)]

    def run(self):
        # Each philosopher will think and eat a couple of times
        for _ in range(2): 
            self.think()
            self.eat()

    def think(self):
        print(f"[{self.name}] is deeply in thought...")
        time.sleep(random.uniform(0.1, 0.3))

    def eat(self):
        print(f"[{self.name}] is getting hungry.")
        
        # Using 'with' acts as a context manager, automatically calling 
        # lock.acquire() on entry and lock.release() on exit, even if errors occur.
        with self.first_fork:
            with self.second_fork:
                print(f"[{self.name}] secured both forks and is EATING.")
                time.sleep(random.uniform(0.1, 0.3))
        
        print(f"[{self.name}] finished eating and dropped the forks.")


In [59]:
num_philosophers = 5
table_forks = [threading.Lock() for _ in range(num_philosophers)]
philosophers = [Philosopher(i, table_forks) for i in range(num_philosophers)]

for p in philosophers:
    p.start()

for p in philosophers:
    p.join()

print("DINNER CONCLUDED SUCCESSFULLY. No deadlocks")

[Philosopher-0] is deeply in thought...
[Philosopher-1] is deeply in thought...
[Philosopher-2] is deeply in thought...
[Philosopher-3] is deeply in thought...
[Philosopher-4] is deeply in thought...
[Philosopher-3] is getting hungry.
[Philosopher-3] secured both forks and is EATING.
[Philosopher-4] is getting hungry.
[Philosopher-0] is getting hungry.
[Philosopher-1] is getting hungry.
[Philosopher-1] secured both forks and is EATING.
[Philosopher-2] is getting hungry.
[Philosopher-3] finished eating and dropped the forks.
[Philosopher-3] is deeply in thought...
[Philosopher-4] secured both forks and is EATING.
[Philosopher-1] finished eating and dropped the forks.[Philosopher-2] secured both forks and is EATING.

[Philosopher-1] is deeply in thought...
[Philosopher-3] is getting hungry.
[Philosopher-2] finished eating and dropped the forks.
[Philosopher-2] is deeply in thought...
[Philosopher-1] is getting hungry.
[Philosopher-1] secured both forks and is EATING.
[Philosopher-4] fini